In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
import random

Mounted at /content/drive


# **Preprocessing**

In [ ]:
raw_data_path = "/content/drive/MyDrive/VVQA_Project/images"
processed_path = "/content/drive/MyDrive/VVQA_Project/processed_dataset"

In [ ]:
import os
import shutil
import random

# Paths

raw_data_path = "/content/drive/MyDrive/VVQA_Project/images"
processed_path = "/content/drive/MyDrive/VVQA_Project/processed_dataset"

# Get images

images = [
    f for f in os.listdir(raw_data_path)
    if f.endswith((".jpg", ".png", ".jpeg"))
]

random.shuffle(images)

n = len(images)


# Safe split (IMPORTANT)
# ensure test is NEVER empty
train_split = int(0.7 * n)
val_split = int(0.85 * n)

# force at least 1 image in test if possible
if n >= 3:
    train_split = max(1, train_split)
    val_split = max(train_split + 1, val_split)

train_imgs = images[:train_split]
val_imgs = images[train_split:val_split]
test_imgs = images[val_split:]


# Create folders
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(processed_path, split, "images"), exist_ok=True)


# Copy function

def copy_imgs(img_list, split_name):
    for img in img_list:
        src = os.path.join(raw_data_path, img)
        dst = os.path.join(processed_path, split_name, "images", img)

        if os.path.exists(src):
            shutil.copy(src, dst)

# Execute
copy_imgs(train_imgs, "train")
copy_imgs(val_imgs, "val")
copy_imgs(test_imgs, "test")


# Check result
print("Train:", len(train_imgs))
print("Val:", len(val_imgs))
print("Test:", len(test_imgs))

Train: 136
Val: 29
Test: 30


# **EfficientNet**

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array

In [ ]:

CSV_PATH    = "/content/drive/MyDrive/VVQA_Project/vqa_dataset.csv"
IMAGES_DIR  = "/content/drive/MyDrive/VVQA_Project/images"
SAVE_PATH   = "/content/drive/MyDrive/VVQA_Project/efficient_features_map.npy"
IMG_SIZE    = (224, 224)

In [ ]:
df = pd.read_csv(CSV_PATH)

print("CSV loaded!")
print(f"   Rows: {len(df)}")
print(f"   Columns: {df.columns.tolist()}")
print(df.head(3))

CSV loaded!
   Rows: 585
   Columns: ['image', 'question', 'answer']
              image                       question  \
0  images/img_0.jpg          What is in the image?   
1  images/img_0.jpg          What object is shown?   
2  images/img_0.jpg  What does the image describe?   

                                answer  
0                                  dog  
1                                  dog  
2  dog running on beach during daytime  


In [ ]:

# EfficientNet layer

base_model = EfficientNetB0(
    weights="imagenet",      # pretrained
    include_top=False,
    pooling= None
)

#  weights(Frozen)
base_model.trainable = False

print(f"\n EfficientNet loaded!")
print(f"   Output shape per image: {base_model.output_shape}")



16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

 EfficientNet loaded!
   Output shape per image: (None, None, None, 1280)


In [ ]:
def extract_features(img_filename):
    """
    Takes an image filename
    Returns a tensor (feature vector) of size 1280
    """
    # Build the full image path
    img_path = os.path.join(IMAGES_DIR, img_filename)

    # Load the image and resize it to the required input size
    img = load_img(img_path, target_size=IMG_SIZE)

    # Convert the image to a NumPy array
    img_array = img_to_array(img)               # shape: (224, 224, 3)

    # Add batch dimension
    img_array = np.expand_dims(img_array, 0)  # shape: (1, 224, 224, 3)

    # Apply EfficientNet preprocessing
    img_array = preprocess_input(img_array)

    # Extract features using the CNN model

    features = base_model.predict(img_array, verbose=0)

    # shape:
    # (1, 7, 7, 1280)

    features = features[0]

    # reshape to patches
    features = features.reshape(-1, 1280)

    # final shape:
    # (49, 1280)

    return features


In [ ]:
features_cache = {}   # image name → extracted features
all_features = []     # features for each row in the CSV

for idx, row in df.iterrows():
    # Extract only the base filename to avoid duplicated 'images/' in path
    img_filename_from_df = row['image']
    img_name = os.path.basename(img_filename_from_df)

    # If the image was already processed, reuse cached features
    if img_name not in features_cache:
        # Corrected: Pass img_name (e.g., 'img_0.jpg') instead of img_filename_from_df (e.g., 'images/img_0.jpg')
        features_cache[img_name] = extract_features(img_name)

    # Append features to the final list
    all_features.append(features_cache[img_name])

    # Print progress every 100 rows
    if (idx + 1) % 100 == 0:
        print(f"   {idx + 1}/{len(df)} done...")

# Convert list of features to numpy array
all_features = np.array(all_features)  # shape: (585, 1280)

print(f"\n Features extracted!")
print(f"   Shape: {all_features.shape}")
# Expected output: (585, 1280)
# 585 samples × 1280 features per image

   100/585 done...
   200/585 done...
   300/585 done...
   400/585 done...


In [ ]:

#  save features

np.save(SAVE_PATH, all_features)
print(f"\n Features saved to: {SAVE_PATH}")


# output
print("\n Sample feature vector (first 10 values):")
print(all_features[0][:10])
print(f"\nMin: {all_features.min():.4f} | Max: {all_features.max():.4f}")



 Features saved to: /content/drive/MyDrive/VVQA_Project/image_features.npy

 Sample feature vector (first 10 values):
[ 0.71620023 -0.0852649  -0.0925699  -0.15894139 -0.13797052  0.24826486
  0.3969992  -0.09331445 -0.11504523  0.89489603]

Min: -0.2631 | Max: 5.1831
